## ASHRAE 140-2023

Section 11: CASE AE205  - Low cooling, Dry-coil 1

In [1]:
import os

# Archived notebook: case files and weather paths are relative to the test folder
if os.path.basename(os.getcwd()) == "old_notebooks":
    os.chdir("..")
import opensimula as osm

sim = osm.Simulation()
pro = sim.new_project("pro")
pro.read_json("cases/case_AE205.json")

Reading project data from dictonary
Reading completed.
Checking project: case_AE205
Checking completed.
Calculating solar direct shadows ...
Simulating case_AE205: ...


100%|██████████| 8760/8760 [00:15<00:00, 576.25step/s, n_iter=5]


In [2]:
import pandas as pd
import psychrolib as sicro
sicro.SetUnitSystem(sicro.SI)
system = pro.component("system")
sys_df=system.variable_dataframe()
space_df=pro.component("spaces_1").variable_dataframe()

print("Q_sensible_zone: ", space_df["system_sensible_heat"][8759]/1000)
print("Q_latent_zone: ", space_df["system_latent_heat"][8759]/1000)
print("Q_sensible_system: ", sys_df["Q_sensible"][8759]/1000)
print("Q_latent_system: ", sys_df["Q_latent"][8759]/1000)

p = system.props["ATM_PRESSURE"]

T_odb = sys_df["T_OA"][8759]
T_owb = sys_df["T_OAwb"][8759]
w_o = sicro.GetHumRatioFromTWetBulb(T_odb,T_owb,p)
v_o = sicro.GetMoistAirVolume(T_odb,w_o,p)
h_o = sicro.GetMoistAirEnthalpy(T_odb,w_o)/1000

T_idb = sys_df["T_MA"][8759]
T_iwb = sys_df["T_MAwb"][8759]
w_i = sicro.GetHumRatioFromTWetBulb(T_idb,T_iwb,p)
v_i = sicro.GetMoistAirVolume(T_idb,w_i,p)
h_i = sicro.GetMoistAirEnthalpy(T_idb,w_i)/1000

T_supply = sys_df["T_SA"][8759]
w_supply = sys_df["w_SA"][8759]/1000
v_supply = sicro.GetMoistAirVolume(T_supply,w_supply,p)
h_supply = sicro.GetMoistAirEnthalpy(T_supply,w_supply)/1000

# Coil output
T_co = sys_df["T_CA"][8759]
w_co = sys_df["w_CA"][8759]/1000
v_co = sicro.GetMoistAirVolume(T_co,w_co,p)
h_co = sicro.GetMoistAirEnthalpy(T_co,w_co)/1000
print("Coil output HR(%): ",sicro.GetRelHumFromHumRatio(T_co,w_co,p)*100)

# Return air
T_ra = space_df["temperature"][8759]
w_ra = space_df["abs_humidity"][8759]/1000
v_ra = sicro.GetMoistAirVolume(T_ra,w_ra,p)
h_ra = sicro.GetMoistAirEnthalpy(T_ra,w_ra)/1000

m_su = sys_df["m_air_flow"][8759]
m_o = sys_df["outdoor_air_fraction"][8759]*m_su
m_r = m_su - m_o

# Return fan outlet
T_rao = sys_df["T_RA"][8759]
w_rao = w_ra
v_rao = sicro.GetMoistAirVolume(T_rao,w_rao,p)
h_rao = sicro.GetMoistAirEnthalpy(T_rao,w_rao)/1000

excel = pd.DataFrame()
excel["OA"] = [T_odb,w_o,v_o*1000,h_o,m_o]
excel["MA"] = [T_idb,w_i,v_i*1000,h_i,m_su]
excel["CO"] = [T_co,w_co,v_co*1000,h_co,m_su]
excel["SA"] = [T_supply,w_supply,v_supply*1000,h_supply,m_su]
excel["RA"] = [T_ra,w_ra,v_ra*1000,h_ra,m_r]
excel["RAO"] = [T_rao,w_rao,v_rao*1000,h_rao,m_r]

excel.to_clipboard()
excel


Q_sensible_zone:  -1.4651666265985521
Q_latent_zone:  -0.5861020575508451
Q_sensible_system:  1.9134245522176303
Q_latent_system:  -0.0
Coil output HR(%):  44.534412764613826


,OA,MA,CO,SA,RA,RAO
0,24.900000,24.055289,18.454370,19.044053,23.332837,23.632940
1,0.004478,0.005861,0.005861,0.005861,0.006551,0.006551
2,850.420784,849.882421,833.866080,835.552325,848.748379,849.607492
3,36.456979,39.120218,33.424514,34.024163,40.141309,40.446870
4,0.113195,0.339587,0.339587,0.339587,0.226392,0.226392


In [3]:
sys_df[["epsilon","epsilon_adp","F_load"]].tail(10)

,epsilon,epsilon_adp,F_load
8750,0.701849,0.999617,0.476895
8751,0.701849,0.999617,0.476895
8752,0.701849,0.999617,0.476895
8753,0.701849,0.999617,0.476895
8754,0.701849,0.999617,0.476895
8755,0.701849,0.999617,0.476895
8756,0.701849,0.999617,0.476895
8757,0.701849,0.999617,0.476895
8758,0.701849,0.999617,0.476895
8759,0.701849,0.999617,0.476895
